Some resources:
1. 2019 LLVM Developers’ Meeting: A. Warzynski “Writing an LLVM Pass: 101” - https://www.youtube.com/watch?v=ar7cJl2aBuU

Typical workflow:

- Set up environment variables so that clang, opt, and llvm-config are found.
- Write the pass source code into a .cpp file.
- Compile the pass into a .dylib.
- Create a test C file, lower it to LLVM IR (.ll).
- Run opt with your pass and capture the output.

In [42]:
import os

# Adjust to your LLVM Homebrew install path
LLVM_HOME = "/opt/homebrew/Cellar/llvm/20.1.8"

os.environ["PATH"] = f"{LLVM_HOME}/bin:" + os.environ["PATH"]
os.environ["DYLD_LIBRARY_PATH"] = f"{LLVM_HOME}/lib:" + os.environ.get("DYLD_LIBRARY_PATH", "")

!which clang
!which opt
!llvm-config --version

/opt/homebrew/Cellar/llvm/20.1.8/bin/clang
/opt/homebrew/Cellar/llvm/20.1.8/bin/opt
20.1.8


In [43]:
pass_code = r"""
#include "llvm/IR/Function.h"
#include "llvm/IR/PassManager.h"
#include "llvm/Passes/PassBuilder.h"
#include "llvm/Passes/PassPlugin.h"
#include "llvm/Support/raw_ostream.h"
#include "llvm/IR/Instructions.h"
#include "llvm/IR/CFG.h"
#include "llvm/Analysis/LoopInfo.h"

#include <map>
#include <string>


using namespace llvm;

struct FuncAnalysisPass : public PassInfoMixin<FuncAnalysisPass> {
  PreservedAnalyses run(Function &F, FunctionAnalysisManager &FAM) {
    // Print to stderr so we definitely see it
    errs() << "\n[FuncAnalysisPass] on function: " << F.getName() << "\n";

    unsigned BBcount = 0, Instcount= 0, loads=0, stores=0;
    std::map<std::string, unsigned> opcodeCounts;
    std::map<std::string, unsigned> callCounts;

    // --- 1. Count BBs, instructions, opcode breakdown, loads/stores, calls ---
    for (auto &BB : F) {
      ++BBcount;
      Instcount += BB.size();
      for (auto &I : BB) {
                opcodeCounts[I.getOpcodeName()]++;

                if (isa<LoadInst>(&I)) loads++;
                if (isa<StoreInst>(&I)) stores++;

                if (auto *callInst = dyn_cast<CallBase>(&I)) {
                    if (Function *calledFunc = callInst->getCalledFunction()) {
                        callCounts[calledFunc->getName().str()]++;
                        errs() << "    Calls function: " << calledFunc->getName() << "\n";
                    }
                }
      }
    }

    // --- 2. Loop Detection using LoopAnalysis ---
    auto &LI = FAM.getResult<llvm::LoopAnalysis>(F);
    outs() << "Loops in function: " << std::distance(LI.begin(), LI.end()) << "\n";
    unsigned loopCount = 0;
    for (auto *L : LI) {
        (void)L; // silence unused variable
        loopCount++;
    }

    // --- 3. Print CFG edges ---
    for (auto &BB : F) {
        errs() << "  BasicBlock " << BB.getName() << " successors: ";
        for (auto *Succ : successors(&BB)) {
            errs() << Succ->getName() << " ";
        }
        errs() << "\n";
    }

    errs() << "  Function: " << F.getName()
           << " | BasicBlocks: " << BBcount
           << " | Instructions: " << Instcount
           << " | Load Instructions: " << loads
           << " | Store Instructions: " << stores 
           << " | Loops: " << loopCount << "\n";

    // --- 5. Instruction breakdown ---
    errs() << "  Instruction breakdown:\n";
        for (auto &entry : opcodeCounts) {
            outs() << "    " << entry.first << ": " << entry.second << "\n";
        }

    // --- 6. Call counts ---
        errs() << "  Calls:\n";
        for (auto &entry : callCounts)
            errs() << "    " << entry.first << ": " << entry.second << "\n";

    errs().flush();
    return PreservedAnalyses::all();
  }
   // Run even if functions have optnone
  static bool isRequired() { return true; }
};


// Register plugin
extern "C" LLVM_ATTRIBUTE_WEAK PassPluginLibraryInfo llvmGetPassPluginInfo() {
  // This line confirms the .dylib was loaded.
  errs() << "FuncAnalysisPass plugin loaded!\n";

  return {
    LLVM_PLUGIN_API_VERSION, "FuncAnalysisPass", "v0.1",
    [](PassBuilder &PB) {
      // 1) Allow `-passes="func-analysis"` at *module* level by adapting to function pass
      PB.registerPipelineParsingCallback(
        [](StringRef Name, ModulePassManager &MPM,
           ArrayRef<PassBuilder::PipelineElement>) {
          if (Name == "func-analysis") {
            errs() << "FuncAnalysisPass added to module pipeline\n";
            MPM.addPass(createModuleToFunctionPassAdaptor(FuncAnalysisPass()));
            return true;
          }
          return false;
        });

      // 2) Also allow `-passes="function(func-analysis)"`
      PB.registerPipelineParsingCallback(
        [](StringRef Name, FunctionPassManager &FPM,
           ArrayRef<PassBuilder::PipelineElement>) {
          if (Name == "func-analysis") {
            errs() << "FuncAnalysisPass added to function pipeline\n";
            FPM.addPass(FuncAnalysisPass());
            return true;
          }
          return false;
        });
    }
  };
}
"""

with open("llvm_passes/FuncAnalysisPass.cpp", "w") as f:
    f.write(pass_code)


In [44]:
c_code = r"""
#include <stdio.h>

int add(int a, int b) {
    return a + b;
}

int sum(int n) {
    int s = 0;
    for (int i = 0; i < n; i++)
        s += i;
    return s;
}

int main() {
    int x = add(1, 2);
    int y = sum(x);
    printf("%d\n", x);
    printf("%d\n", y);
    return 0;
}
"""

with open("llvm_passes/add.c", "w") as f:
    f.write(c_code)

In [45]:
!clang -emit-llvm -S -O0 llvm_passes/add.c -o llvm_passes/add.ll

In [46]:
!clang++ -std=c++17 -fPIC -shared llvm_passes/FuncAnalysisPass.cpp -o llvm_passes/FuncAnalysisPass.dylib \
    `llvm-config --cxxflags --ldflags --system-libs --libs core passes`


In [47]:
!opt -load-pass-plugin=./llvm_passes/FuncAnalysisPass.dylib -passes="function(func-analysis)" -disable-output llvm_passes/add.ll

FuncAnalysisPass plugin loaded!
FuncAnalysisPass added to function pipeline

[FuncAnalysisPass] on function: add
Loops in function: 0
  BasicBlock  successors: 
  Function: add | BasicBlocks: 1 | Instructions: 8 | Load Instructions: 2 | Store Instructions: 2 | Loops: 0
  Instruction breakdown:
    add: 1
    alloca: 2
    load: 2
    ret: 1
    store: 2
  Calls:

[FuncAnalysisPass] on function: sum
Loops in function: 1
  BasicBlock  successors:  
  BasicBlock  successors:   
  BasicBlock  successors:  
  BasicBlock  successors:  
  BasicBlock  successors: 
  Function: sum | BasicBlocks: 5 | Instructions: 22 | Load Instructions: 6 | Store Instructions: 5 | Loops: 1
  Instruction breakdown:
    add: 2
    alloca: 3
    br: 4
    icmp: 1
    load: 6
    ret: 1
    store: 5
  Calls:

[FuncAnalysisPass] on function: main
    Calls function: add
    Calls function: sum
    Calls function: printf
    Calls function: printf
Loops in function: 0
  BasicBlock  successors: 
  Function: main | Bas